# 根据历史评分推荐电影

深度学习期末小组作业

[项目仓库](https://github.com/xunan23333-bit/movielens-sequence-rec)

## 1. 按人、按时间切分评分记录

- `ratings.csv` 提供用户、电影、评分和时间。用 `u`、`i` 把原编号换成连续编号，电影编号 0 留给序列空位。
- `rank` 从每人的最新记录倒数。最近 5 条进 `test`，再往前 5 条进 `valid`，更早的进 `train`。
- `original` 存每个人的训练历史。`train_seen` 标记训练中评过的电影，`seen` 标记全部已知交互，供评估和推荐过滤使用。


In [1]:
from pathlib import Path
import copy
import json
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score

OUT = Path("out")
OUT.mkdir(exist_ok=True)
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv").set_index("movieId")
user_ids = np.sort(ratings.userId.unique())
movie_ids = np.sort(ratings.movieId.unique())
ratings["u"] = np.searchsorted(user_ids, ratings.userId)
ratings["i"] = np.searchsorted(movie_ids, ratings.movieId) + 1
ratings = ratings.sort_values(["u", "timestamp"], kind="mergesort")
rank = ratings.groupby("u").cumcount(ascending=False)
train = ratings[rank >= 10]
valid = ratings[(rank >= 5) & (rank < 10)]
test = ratings[rank < 5]
U, I = len(user_ids), len(movie_ids) + 1
seen = np.zeros((U, I), dtype=bool)
seen[ratings.u, ratings.i] = True
train_seen = np.zeros_like(seen)
train_seen[train.u, train.i] = True

def histories(frame):
    return [g[["i", "rating", "timestamp"]].to_numpy() for _, g in frame.groupby("u")]

original = histories(train)
assert len(train) + len(valid) + len(test) == len(ratings)
assert (train.groupby("u").timestamp.max() <= valid.groupby("u").timestamp.min()).all()
assert (valid.groupby("u").timestamp.max() <= test.groupby("u").timestamp.min()).all()
print(dev, torch.__version__, len(train), len(valid), len(test))

cuda 2.12.0+cu126 94736 3050 3050


## 2. 把历史做成训练样本，再固定评估候选

- `inputs()` 只取最近 `width` 条，靠右放进同样宽的数组。宽度为 4 时，A→B 放成 [0, 0, A, B]。`weights` 把空位乘成 0，还可以让较早的记录权重小一些。
- `training_data()` 把 A→B→C 切成「A 猜 B」和「A、B 猜 C」。`choices` 第 0 列是真目标，后面 100 次负例抽取允许重复，但都避开该用户的训练交互。
- `candidates()` 为每人固定 5 个真目标和 100 个不同的未交互电影。`row - 1` 把电影内部编号变成打分矩阵的列下标。`select` 和 `review` 是两组用户，分别用于选配置和复核。


In [2]:
def inputs(items, width, decay):
    x = np.zeros((len(items), width), dtype=np.int64)
    r = np.full_like(x, 10)
    dates = np.zeros_like(x)
    for row, history in enumerate(items):
        h = history[-width:]
        x[row, -len(h):] = h[:, 0]
        r[row, -len(h):] = np.rint(h[:, 1] * 2).astype(int) - 1
        dates[row, -len(h):] = h[:, 2]
    age = np.zeros_like(dates)
    for col in range(width - 2, -1, -1):
        age[:, col] = np.where(dates[:, col] == dates[:, col + 1],
                               age[:, col + 1], width - col - 1)
    weights = ((decay ** age) * (x != 0)).astype(np.float32)
    movies = torch.as_tensor(x, device=dev)
    ratings = torch.as_tensor(r, device=dev)
    weights = torch.as_tensor(weights, device=dev)
    return movies, ratings, weights

def training_data(sequences, width, decay):
    prefix = []
    users = []
    target = []
    known = train_seen.copy()
    for u, h in enumerate(sequences):
        known[u, h[:, 0].astype(int)] = True
        for end in range(1, len(h)):
            prefix.append(h[max(0, end-width):end])
            users.append(u)
            target.append(int(h[end, 0]))
    rows = np.array(users)[:, None]
    rng = np.random.default_rng(612)
    negatives = rng.integers(1, I, size=(len(target), 100))
    bad = known[rows, negatives]
    while bad.any():
        negatives[bad] = rng.integers(1, I, size=bad.sum())
        bad = known[rows, negatives]
    assert not known[rows, negatives].any()
    choices = np.column_stack((target, negatives))
    history = inputs(prefix, width, decay)
    return history, torch.as_tensor(choices, device=dev)

rng = np.random.default_rng(42)
# Advance past the original baseline draws to preserve the fixed evaluation lists.
for u, h in enumerate(original):
    pool = np.flatnonzero(~train_seen[u, 1:]) + 1
    for _ in h[1:]:
        rng.choice(pool, 1, replace=False)

def candidates(frame):
    output = []
    for u, g in frame.groupby("u"):
        positive = g.i.to_numpy()
        negative = rng.choice(np.flatnonzero(~seen[u, 1:]) + 1, 100, replace=False)
        row = np.r_[positive, negative]
        assert len(positive) == 5 and len(np.unique(row)) == 105
        output.append(row - 1)
    return np.array(output)

test_items = candidates(test)
valid_items = candidates(valid)
order = np.random.default_rng(58123).permutation(U)
select, review = np.sort(order[:305]), np.sort(order[305:])

## 3. 给每部电影准备题材和训练评分统计

- `genre` 把题材拆成多列，喜剧和爱情可以同时为 1。`np.pad()` 给电影编号 0 补一行空位。
- `count` 和 `total` 只查训练记录。平滑均分相当于先给每部电影加 10 条全体均分的评分，样本少的电影因此更靠近全体均分。
- `side_tables` 存四种配置：不用侧特征、只用题材、只用评分统计、两种都用。建模型时用 `side` 选出对应的表。


In [3]:
genre = movies.loc[movie_ids, "genres"].str.get_dummies(sep="|").to_numpy()
genre = np.pad(genre, ((1, 0), (0, 0)))
count = np.bincount(train.i, minlength=I)
total = np.bincount(train.i, weights=train.rating, minlength=I)
mean = train.rating.mean()
smooth_mean = (total + 10 * mean) / (count + 10)
rating_feature = (smooth_mean - mean) / 2
count_feature = np.log1p(count) / np.log1p(count.max())
stats = np.column_stack((rating_feature, count_feature))
side_tables = {"none": None, "genres": genre, "stats": stats,
               "both": np.column_stack((genre, stats))}

## 4. 让 GRU 和查询注意力各算一个用户向量

- `items()` 取出电影的 64 维向量，有侧特征时先把它映射到 64 维再相加。`encode()` 还给历史电影加上评分向量，并乘近期权重。
- GRU 按顺序更新状态，取 `states[:, -1]`。查询注意力加上位置向量，由可训练的 `query` 读取历史，`padding` 屏蔽空位。得到的表示还要经过两次残差连接与归一化。
- `forward()` 用同一个用户向量给每个候选打分。若用户向量是 [2, 1]、电影向量是 [3, 4]，点积就是 2×3 + 1×4 = 10。


In [4]:
class SequenceModel(nn.Module):
    def __init__(self, kind, width, side):
        super().__init__()
        self.movie = nn.Embedding(I, 64, padding_idx=0)
        self.rating = nn.Embedding(11, 64, padding_idx=10)
        if side is None:
            self.register_buffer("side", None)
            self.project = None
        else:
            self.register_buffer("side", torch.tensor(side, dtype=torch.float32))
            self.project = nn.Linear(side.shape[1], 64, bias=False)
        self.kind = kind
        if kind == "GRU":
            self.gru = nn.GRU(64, 64, batch_first=True)
        else:
            self.position = nn.Embedding(width + 1, 64)
            self.query = nn.Parameter(torch.zeros(1, 1, 64))
            self.attention = nn.MultiheadAttention(64, 2, batch_first=True, dropout=.15)
            self.norm1 = nn.LayerNorm(64)
            self.norm2 = nn.LayerNorm(64)
            self.ff = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64))
        self.drop = nn.Dropout(.15)
        nn.init.normal_(self.movie.weight, std=.12)
        nn.init.zeros_(self.movie.weight[:1])
        nn.init.normal_(self.rating.weight, std=.03)
        nn.init.zeros_(self.rating.weight[10:])

    def items(self, ids):
        vector = self.movie(ids)
        if self.project is not None:
            vector = vector + self.project(self.side[ids])
        return vector

    def encode(self, x, r, weights):
        history = (self.items(x) + self.rating(r)) * weights.unsqueeze(-1)
        if self.kind == "GRU":
            states, _ = self.gru(history)
            return self.drop(states[:, -1])
        positions = self.position(torch.arange(x.size(1) + 1, device=x.device))
        query = (self.query + positions[-1]).expand(x.size(0), -1, -1)
        source = torch.cat((history + positions[:-1], query), dim=1)
        padding = F.pad(x == 0, (0, 1), value=False)
        update, _ = self.attention(query, source, source, key_padding_mask=padding, need_weights=False)
        state = self.norm1(query + self.drop(update))
        return self.norm2(state + self.drop(self.ff(state)))[:, 0]

    def forward(self, x, r, w, choices):
        user = self.encode(x, r, w)
        candidates = self.items(choices)
        return (user[:, None] * candidates).sum(-1)

## 5. 训练模型，并用同一套函数算指标

- `score_all()` 给所有电影打分。`evaluate()` 取出固定候选，计算 AUC、MAP@10 和 NDCG@10。全量 NDCG@10 会屏蔽其余已知交互，保留当前 5 个目标。
- `fit()` 每批给候选打分。真目标在第 0 列，交叉熵的正确类别全填 0。`backward()` 算梯度，`opt.step()` 更新模型。
- 每四轮用 `select` 的全量 NDCG@10 检查一次，最后恢复最好的一轮。函数返回模型、运行记录，以及 `review` 每位用户的采样 NDCG@10，最后一项用于 GAN 对照。


In [5]:
discount = 1 / np.log2(np.arange(2, 12))
ideal = discount[:5].sum()

@torch.no_grad()
def score_all(model, history):
    model.eval()
    return (model.encode(*history) @ model.items(torch.arange(1, I, device=dev)).T).cpu().numpy()

def evaluate(scores, candidate, users):
    rows = np.arange(len(users))[:, None]
    ids = candidate[users]
    positive = ids[:, :5]
    values = scores[users[:, None], ids]
    labels = np.broadcast_to(np.r_[np.ones(5), np.zeros(100)], values.shape)
    top = np.argsort(-values, axis=1, kind="stable")[:, :10]
    hits = labels[rows, top]
    ap = (hits * hits.cumsum(1) / np.arange(1, 11)).sum(1) / 5
    ndcg = hits @ discount / ideal
    full = scores[users].copy()
    mask = seen[users, 1:].copy()
    mask[rows, positive] = False
    full[mask] = -np.inf
    top = np.argsort(-full, axis=1, kind="stable")[:, :10]
    full_ndcg = (top[:, :, None] == positive[:, None, :]).any(2) @ discount / ideal
    return {"AUC": float(roc_auc_score(labels.ravel(), values.ravel())), "MAP10": float(ap.mean()),
            "NDCG10": float(ndcg.mean()), "FullNDCG10": float(full_ndcg.mean())}, ndcg

# A perfect top five gives NDCG@10 = 1.
assert np.isclose(discount[:5].sum() / ideal, 1)
 
def fit(kind, sequences, config, seed):
    width, decay, feature = config
    (movies, ratings, recent_weights), pairs = training_data(sequences, width, decay)
    history = inputs(original, width, decay)
    torch.manual_seed(seed)
    model = SequenceModel(kind, width, side_tables[feature]).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=.002, weight_decay=1e-4)
    best, best_epoch = -1, 0
    start = time.perf_counter()
    for epoch in range(1, 25):
        model.train()
        order = torch.randperm(len(pairs), device=dev)
        for batch in order.split(768):
            opt.zero_grad(set_to_none=True)
            logits = model(movies[batch], ratings[batch], recent_weights[batch], pairs[batch])
            target = torch.zeros(len(batch), dtype=torch.long, device=dev)
            loss = F.cross_entropy(logits, target)
            loss.backward()
            opt.step()
        if epoch % 4 == 0:
            metrics, _ = evaluate(score_all(model, history), valid_items, select)
            if metrics["FullNDCG10"] > best:
                best, best_epoch = metrics["FullNDCG10"], epoch
                weights = copy.deepcopy(model.state_dict())
    model.load_state_dict(weights)
    scores = score_all(model, history)
    chosen, _ = evaluate(scores, valid_items, select)
    audited, users = evaluate(scores, valid_items, review)
    result = dict(kind=kind, config=config, seed=seed, epoch=best_epoch,
                  seconds=round(time.perf_counter()-start, 1),
                  params=sum(p.numel() for p in model.parameters()),
                  selection=chosen, review=audited)
    print(kind, config, seed, best_epoch, result["seconds"], audited, flush=True)
    return model, result, users

## 6. 用验证集选长度、衰减和电影特征

- `record()` 跑一次训练并保存记录。`best_config()` 只用种子 42 的 Query 在 `select` 组上的全量 NDCG@10 来做选择。
- 先比 40 条和 20 条，再看选出的长度加衰减是否更好，最后比较不同的侧特征。`winner` 保存锁定的配置。
- 种子 7 用来重复运行。GRU 和 Query 基础版都用 40 条、等权和无侧特征，方便对比两种历史编码方法。
- 本页存档输出选中 40 条、衰减和两类侧特征。用同样代码和种子重新运行时，也观测到 20 条、等权和两类侧特征被选中。复跑数字可能和存档输出有差别。


In [6]:
baseline = (40, 1., "none")
checks = [baseline, (20, 1., "none")]
runs, trained = [], {}

def record(key, kind, config, seed, data=original):
    model, row, scores = fit(kind, data, config, seed)
    row["arm"] = key
    runs.append(row)
    (OUT / "progress.json").write_text(json.dumps(runs, indent=2), encoding="utf8")
    return model, scores

def best_config(options):
    validation = {}
    for row in runs:
        if row["kind"] == "Query" and row["seed"] == 42:
            config = tuple(row["config"])
            validation[config] = row["selection"]["FullNDCG10"]
    return max(options, key=validation.get)

for config in checks:
    trained[config] = record("main", "Query", config, 42)
width = best_config(checks)[0]
weighted = (width, .95, "none")
trained[weighted] = record("main", "Query", weighted, 42)
history_config = best_config(checks + [weighted])
feature_checks = [history_config]
for feature in ("genres", "stats", "both"):
    config = (*history_config[:2], feature)
    trained[config] = record("main", "Query", config, 42)
    feature_checks.append(config)
winner = best_config(feature_checks)
query2, original2 = record("main", "Query", winner, 7)
if winner == baseline:
    baseline2 = query2
else:
    baseline2, _ = record("main", "Query", baseline, 7)
gru, _ = record("main", "GRU", baseline, 42)
gru2, _ = record("main", "GRU", baseline, 7)
print("Locked config:", winner)

Query (40, 1.0, 'none') 42 20 17.4 {'AUC': 0.9191820155872077, 'MAP10': 0.5730143117356231, 'NDCG10': 0.6923579227013694, 'FullNDCG10': 0.09249062563840586}


Query (20, 1.0, 'none') 42 20 17.6 {'AUC': 0.9170699704380543, 'MAP10': 0.559094977881863, 'NDCG10': 0.6787938950394535, 'FullNDCG10': 0.08232179908771055}


Query (40, 0.95, 'none') 42 20 17.9 {'AUC': 0.9161719322762698, 'MAP10': 0.562335415040333, 'NDCG10': 0.6826269310940152, 'FullNDCG10': 0.09072637020103247}


Query (40, 0.95, 'genres') 42 20 19.9 {'AUC': 0.9241667078742274, 'MAP10': 0.6089018995576373, 'NDCG10': 0.7271506784777995, 'FullNDCG10': 0.10643747439545352}


Query (40, 0.95, 'stats') 42 20 18.5 {'AUC': 0.858651083042193, 'MAP10': 0.5559914129586261, 'NDCG10': 0.6796825991033266, 'FullNDCG10': 0.08449025989750893}


Query (40, 0.95, 'both') 42 20 18.5 {'AUC': 0.9004580704111798, 'MAP10': 0.5957410876919074, 'NDCG10': 0.7112099652868691, 'FullNDCG10': 0.09779687130627768}


Query (40, 0.95, 'both') 7 16 19.4 {'AUC': 0.9005670088685837, 'MAP10': 0.5981678376268541, 'NDCG10': 0.7147022990014831, 'FullNDCG10': 0.10464829367090932}


Query (40, 1.0, 'none') 7 24 17.1 {'AUC': 0.9194542542327332, 'MAP10': 0.5696432474629196, 'NDCG10': 0.687417072272396, 'FullNDCG10': 0.08831065653781114}


GRU (40, 1.0, 'none') 42 24 13.4 {'AUC': 0.9124001934963719, 'MAP10': 0.5334876398646891, 'NDCG10': 0.6550562355448575, 'FullNDCG10': 0.08849445693391195}


GRU (40, 1.0, 'none') 7 20 13.5 {'AUC': 0.9072570599301265, 'MAP10': 0.53664012490242, 'NDCG10': 0.656069778778095, 'FullNDCG10': 0.06968554622546913}


Locked config: (40, 0.95, 'both')


## 7. 做 GAN 增强和等量热门对照

- `profiles` 每行对应一个用户，每列对应训练集中一部热门片。生成器把 32 个随机数变成一份评分画像，判别器练习辨认真实画像和生成画像。
- `augment()` 给历史不足 60 条的用户挑一些训练中没评过的电影。`gan_set` 由生成器挑片，`hot_set` 从同一热门池随机抽片。两组给同一用户增加的条数、评分、时间都相同。
- 三组训练数据用同一推荐模型、两个种子，在 `review` 分开看短历史和长历史。短历史差值区间包含 0，这次没有确认 GAN 能提高指标。


In [7]:
popular = np.argsort(-count[1:], kind="stable")[:200] + 1
column = {item: j for j, item in enumerate(popular)}
profiles = np.zeros((U, len(popular)), dtype=np.float32)
for row in train.itertuples():
    if row.i in column:
        profiles[row.u, column[row.i]] = row.rating / 5
torch.manual_seed(42)
real = torch.tensor(profiles, device=dev)
generator = nn.Sequential(nn.Linear(32, 128), nn.LeakyReLU(.2),
                          nn.Linear(128, 200), nn.Sigmoid()).to(dev)
critic = nn.Sequential(nn.Linear(200, 128), nn.LeakyReLU(.2), nn.Linear(128, 1)).to(dev)
g_opt = torch.optim.Adam(generator.parameters(), lr=.0002, betas=(.5, .999))
d_opt = torch.optim.Adam(critic.parameters(), lr=.0002, betas=(.5, .999))
bce = nn.BCEWithLogitsLoss()
for epoch in range(400):
    for batch in torch.randperm(U, device=dev).split(64):
        z = torch.randn(len(batch), 32, device=dev)
        yes = torch.ones((len(batch), 1), device=dev)
        d_opt.zero_grad()
        d_loss = bce(critic(real[batch]), yes) + bce(critic(generator(z).detach()), 1 - yes)
        d_loss.backward()
        d_opt.step()
        g_opt.zero_grad()
        g_loss = bce(critic(generator(z)), yes)
        g_loss.backward()
        g_opt.step()

def augment():
    rng = np.random.default_rng(1121)
    gan, hot = [], []
    with torch.no_grad():
        fake = (generator(torch.randn(U, 5, 32, device=dev)) * 5).cpu().numpy()
    for u, h in enumerate(original):
        available = [j for j in range(200) if not train_seen[u, popular[j]]]
        selected = []
        for j in available:
            rating = float(fake[u, :, j].max())
            if rating >= 4:
                selected.append((popular[j], rating))
        rng.shuffle(selected)
        selected = selected[:max(60-len(h), 0)]
        other = rng.choice(popular[available], len(selected), replace=False)
        dates = np.arange(1, len(selected)+1) + h[-1, 2]
        scores = [r for _, r in selected]
        gan.append(np.vstack((h, np.column_stack(([i for i, _ in selected], scores, dates)))))
        hot.append(np.vstack((h, np.column_stack((other, scores, dates)))))
    return gan, hot

gan_set, hot_set = augment()
added = sum(len(a)-len(b) for a, b in zip(gan_set, original))
assert all(len(a) == len(b) and np.array_equal(a[:, 1:], b[:, 1:]) for a, b in zip(gan_set, hot_set))
short = np.array([len(original[u]) < 60 for u in review])
groups = {"original": [trained[winner][1], original2], "gan": [], "popular": []}
for seed in (42, 7):
    for name, data in (("gan", gan_set), ("popular", hot_set)):
        _, scores = record(name, "Query", winner, seed, data)
        groups[name].append(scores)
gan_result = {}
for name, values in groups.items():
    averages = np.mean(values, axis=0)
    gan_result[name] = [float(averages[short].mean()), float(averages[~short].mean())]
delta = np.mean(groups["gan"], axis=0)[short] - np.mean(groups["original"], axis=0)[short]
bootstrap = np.random.default_rng(1729).choice(delta, size=(2000, len(delta)))
interval = np.quantile(bootstrap.mean(1), [.025, .975]).tolist()
print("GAN", added, gan_result, interval)

Query (40, 0.95, 'both') 42 24 19.4 {'AUC': 0.9010575759204514, 'MAP10': 0.592632058287796, 'NDCG10': 0.7106028433984682, 'FullNDCG10': 0.10020842164774849}


Query (40, 0.95, 'both') 42 20 20.1 {'AUC': 0.8998317656543939, 'MAP10': 0.5885820973198023, 'NDCG10': 0.7061202300947591, 'FullNDCG10': 0.10153295171049313}


Query (40, 0.95, 'both') 7 20 20.2 {'AUC': 0.902246019887127, 'MAP10': 0.5928004163413999, 'NDCG10': 0.708407775478083, 'FullNDCG10': 0.10205531333073223}


Query (40, 0.95, 'both') 7 20 20.2 {'AUC': 0.9035970115560334, 'MAP10': 0.586991152745251, 'NDCG10': 0.7053742965631599, 'FullNDCG10': 0.103604988675147}


GAN 8650 {'original': [0.7735536067994117, 0.6452047889532526], 'gan': [0.771588311708219, 0.6400930638447974], 'popular': [0.7662702341153668, 0.6380792196024903]} [-0.018397448174113786, 0.015845475618046962]


## 8. 报告测试结果，再给每位用户推荐十部片

- `groups_test` 中两种模型对应三行，Query 基础版和选定版是同一结构的两种配置。先分别算种子 42、7 的测试指标，再求均值，写到 `out/results.json`。
- 输出推荐时，用种子 42 的选定模型读入现有的全部交互，打分后用 `seen` 屏蔽已经评过的片，再取最高的十部。
- `movie_ids[top]` 把列下标换回原电影 ID，再查片名。三条 `assert` 检查每人 10 条、没有重复和已评电影。完整列表写到 `out/top10.csv`。


In [8]:
chosen_model = trained[winner][0]
groups_test = {"GRU": (baseline, [gru, gru2]),
               "QueryBase": (baseline, [trained[baseline][0], baseline2]),
               "QuerySelected": (winner, [chosen_model, query2])}
tests = {}
for name, (config, models) in groups_test.items():
    h = inputs(original, *config[:2])
    values = []
    for model in models:
        scores = score_all(model, h)
        metrics, _ = evaluate(scores, test_items, np.arange(U))
        values.append(metrics)
    average = {key: float(np.mean([row[key] for row in values])) for key in values[0]}
    tests[name] = {"seeds": values, "mean": average}
report = dict(winner=winner, runs=runs, test=tests, gan=gan_result,
              added=added, short=int(short.sum()), gan_ci=interval,
              device=torch.cuda.get_device_name() if dev == "cuda" else dev, torch=str(torch.__version__))
(OUT / "results.json").write_text(json.dumps(report, indent=2), encoding="utf8")
print(pd.DataFrame({k: v["mean"] for k, v in tests.items()}).T.round(4))

scores = score_all(chosen_model, inputs(histories(ratings), *winner[:2]))
scores[seen[:, 1:]] = -np.inf
top = np.argsort(-scores, axis=1, kind="stable")[:, :10]
ids = movie_ids[top].ravel()
recommend = pd.DataFrame(dict(userId=np.repeat(user_ids, 10), rank=np.tile(np.arange(1, 11), U),
                              movieId=ids, title=movies.loc[ids, "title"].to_numpy()))
assert recommend.groupby("userId").size().eq(10).all()
assert not recommend.duplicated(["userId", "movieId"]).any()
assert recommend.merge(ratings[["userId", "movieId"]], on=["userId", "movieId"]).empty
recommend.to_csv(OUT / "top10.csv", index=False, encoding="utf-8-sig")
print(len(recommend), "recommendations")
display(recommend[recommend.userId.isin([1, 53])])

                  AUC   MAP10  NDCG10  FullNDCG10
GRU            0.8841  0.4476  0.5752      0.0491
QueryBase      0.8934  0.4750  0.6026      0.0513
QuerySelected  0.8717  0.5034  0.6284      0.0613


6100 recommendations


,userId,rank,movieId,title
0,1,1,2706,American Pie (1999)
1,1,2,1479,"Saint, The (1997)"
2,1,3,2539,Analyze This (1999)
3,1,4,1614,In & Out (1997)
4,1,5,2724,Runaway Bride (1999)
5,1,6,3555,U-571 (2000)
6,1,7,135,Down Periscope (1996)
7,1,8,372,Reality Bites (1994)
8,1,9,2572,10 Things I Hate About You (1999)
9,1,10,3948,Meet the Parents (2000)
